In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd

import helpers

SEEDS = range(32, 37)
CONDITIONS = ['none', 'tabular', 'tabular-text', 'tabular-image', 'tabular-image-text']
TEXT_WIDTHS = [50]  # Main thesis matrix; historical width sweep stored separately.

gender_test = helpers.load_gender_test()
male = (gender_test == 0)
female = (gender_test == 1)

In [ ]:
class MidFusion(nn.Module):
    def __init__(self, tabular_size, text_size, image_size, text_width):
        super().__init__()
        self.tabular = nn.Linear(tabular_size, tabular_size)
        self.text = nn.Linear(text_size, text_width)
        self.image = nn.Linear(image_size, image_size)

        self.fc1 = nn.Linear(tabular_size + text_width + image_size, 40)
        self.fc2 = nn.Linear(40, 20)
        self.fc3 = nn.Linear(20, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, tabular, text, image):
        # each modality gets its own layer
        tabular = torch.relu(self.tabular(tabular))
        text = torch.relu(self.text(text))
        image = torch.relu(self.image(image))
        cat = torch.cat((tabular, text, image), 1)

        cat = torch.relu(self.fc1(cat))
        cat = self.dropout(cat)
        cat = torch.relu(self.fc2(cat))
        return torch.sigmoid(self.fc3(cat))

In [ ]:
rows = []

for text_width in TEXT_WIDTHS:
    for condition in CONDITIONS:
        (labels_train_biased, labels_train_blind, labels_test_biased, labels_test_blind,
         tabular_train, tabular_test, text_train, text_test, faces_train, faces_test) = helpers.build_features(condition)

        for seed in SEEDS:
            predictions, model = helpers.train_and_evaluate(
                lambda: MidFusion(tabular_train.shape[1], text_train.shape[1], faces_train.shape[1], text_width),
                [tabular_train, text_train, faces_train],
                [tabular_test, text_test, faces_test],
                labels_train_biased,
                seed,
            )

            parity, top100_female = helpers.computeDemographicParity(predictions, gender_test)
            equal_opportunity, n_selected = helpers.computeEqualityOfOpportunity(predictions, gender_test, labels_train_blind, labels_test_blind, p = 75)

            rows.append({
                'condition': condition,
                'fusion': 'mid',
                'seed': seed,
                'mae_biased': helpers.computeMAE(predictions, labels_test_biased),
                'mae_blind': helpers.computeMAE(predictions, labels_test_blind),
                'kl': helpers.computeKL(predictions[male], predictions[female]),
                'parity': parity,
                'top100_female': top100_female,
                'equal_opportunity': equal_opportunity,
                'n_selected': n_selected,
                'ndcg': helpers.computeNDCG(predictions, labels_test_blind),
                'text_width': text_width
            })

        print(condition, 'done')

results = pd.DataFrame(rows)
results.to_csv('../results/results_mid.csv', index=False)
results.groupby(['text_width', 'condition'], sort=False)[['mae_biased', 'mae_blind', 'kl', 'parity', 'top100_female','equal_opportunity', 'n_selected', 'ndcg']].agg(['mean', 'std'])